# 重算 Span Recall（SR_para）—— WI-IAT 2026 Table 5 修正

**目的**：把 Table 5 里原来的 "Span Recall" 替换成真正与 Stage 2 无关的
**段落级金标准跨度可达率**（paragraph-level gold-span reachability），
使 "hard upper bound" 这句话在数学上真正成立。

## 定义

对每一条金标准跨度 (article, tech, gs, ge)，判断它所在文章里**是否存在**
一个被 Stage 1 判为正例的段落 [ps, pe] 与之重叠：

- **SR_para（技术无关）**：只要任一被判正例的段落覆盖该跨度即算可达。
  这是 **S2-C 的真正硬上界**——因为 S2-C 联合预测全部 23 个技术，
  不受 Stage 1 技术预测的约束，只受"段落是否被送进 Stage 2"的约束。
- **SR_tech（技术匹配）**：必须存在一个被判正例、**且技术匹配**的段落覆盖该跨度。
  这是 **S2-A 的上界**——因为 S2-A 只定位 Stage 1 给它的那个技术。

理论上应满足：`S2-A_recall ≤ SR_tech ≤ SR_para`，且 `S2-C_recall ≤ SR_para`。

> 论文 Table 5 用 **SR_para** 这一列（技术无关），并改写为
> "the fraction of gold spans whose containing paragraph is retrieved by Stage 1"，
> 这样它对任意 Stage 2 方法都成立。

**运行**：从上到下依次执行即可，最后一格打印 9 个格子 + 汇总表。

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory of this repository
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 Task 3 development data
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirectories

# CheckThat! 2024 Task 3 data
CHECKTHAT_DIR = "your/path/here"

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


## 步骤 1：路径配置（与 43 notebook 完全一致）

In [ ]:
import os
from collections import defaultdict

BASE_DIR  = BASE  # Set BASE in the configuration cell
LLM_DIR   = f"{BASE_DIR}/results/LLM_tests"
MODEL_DIR = f"{BASE_DIR}/results/model_tests"

# ── Stage 1 段落级预测文件（路径以 实验结果汇总 附录为准；EN Prompt-A/Iter-Ens 带 results_en_ 前缀）──
PARAGRAPH_FILES = {
    ("en", "Iter-Ens"): f"{LLM_DIR}/results_en_results_semeval_task3_dev_en_voting_aggressive_all.tsv",
    ("en", "Prompt-A"): f"{LLM_DIR}/results_en_results_semeval_task3_dev_en_context_all.tsv",
    ("en", "Sup-FT"):   f"{MODEL_DIR}/semeval_task3_en_dev_predictions_official_format.txt",
    ("po", "Iter-Ens"): f"{LLM_DIR}/semeval2023_task3_dev_results_po_po_voting_aggressive_all.tsv",
    ("po", "Prompt-A"): f"{LLM_DIR}/semeval2023_task3_dev_results_po_po_context_all.tsv",
    ("po", "Sup-FT"):   f"{MODEL_DIR}/semeval_task3_po_dev_predictions_official_format.txt",
    ("ru", "Iter-Ens"): f"{LLM_DIR}/semeval2023_task3_dev_results_ru_ru_voting_aggressive_all.tsv",
    ("ru", "Prompt-A"): f"{LLM_DIR}/semeval2023_task3_dev_results_ru_ru_context_all.tsv",
    ("ru", "Sup-FT"):   f"{MODEL_DIR}/semeval_task3_ru_dev_predictions_official_format.txt",
}

# ── 金标准跨度文件（按语言）──
def gold_path(lang):
    return f"{BASE_DIR}/data_analy/overlap_analysis_results/{lang}_overlap_articles/labels-subtask-3-spans.txt"

LANGS   = ["en", "po", "ru"]
METHODS = ["Sup-FT", "Prompt-A", "Iter-Ens"]

# 路径验证
print("📁 路径验证：")
for lang in LANGS:
    gp = gold_path(lang)
    print(f"  {'✅' if os.path.exists(gp) else '❌'} gold[{lang}]: {gp}")
    for method in METHODS:
        pf = PARAGRAPH_FILES[(lang, method)]
        print(f"  {'✅' if os.path.exists(pf) else '❌'} {method:9s}[{lang}]: {os.path.basename(pf)}")

## 步骤 2：加载函数（自包含，不依赖 43 notebook）

In [ ]:
def load_spans_tsv(path):
    """读取 4 列 TSV: article_id \t technique \t start \t end。
    返回 dict: {article_id: [(technique, start, end), ...]}。
    对金标准与 Stage 1 预测通用；article_id 统一去除 'article' 前缀，
    以对齐 Sup-FT（article+数字）与金标准/LLM（纯数字）两种格式。"""
    by_article = defaultdict(list)
    n_rows = 0
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) < 4:
                continue
            try:
                aid  = str(parts[0]).replace("article", "").strip()
                tech = str(parts[1]).strip()
                s    = int(parts[2])
                e    = int(parts[3])
            except ValueError:
                continue  # 跳过表头/脏行
            if e < s:
                s, e = e, s
            by_article[aid].append((tech, s, e))
            n_rows += 1
    return by_article, n_rows


def overlaps(a_s, a_e, b_s, b_e):
    """半开区间重叠判定（与 span F1 的 IoU>0 口径一致）。"""
    return a_s < b_e and b_s < a_e


print("✅ Loading function ready")

## 步骤 3：SR 计算函数

In [ ]:
def compute_span_recall(gold_by_article, pred_by_article):
    """返回 SR_para（技术无关）与 SR_tech（技术匹配）。

    SR_para = (存在任一被判正例段落覆盖的金标准跨度数) / (金标准跨度总数)
    SR_tech = (存在技术匹配且被判正例段落覆盖的金标准跨度数) / (金标准跨度总数)
    """
    total_gold       = 0
    reach_para       = 0
    reach_tech       = 0

    for aid, golds in gold_by_article.items():
        preds = pred_by_article.get(aid, [])
        # 该文章被 Stage 1 判为正例的段落区间（技术无关，去重不必要）
        pred_intervals_any  = [(ps, pe) for (_, ps, pe) in preds]
        # 按技术索引的段落区间
        pred_intervals_tech = defaultdict(list)
        for (ptech, ps, pe) in preds:
            pred_intervals_tech[ptech].append((ps, pe))

        for (gtech, gs, ge) in golds:
            total_gold += 1
            # 技术无关可达
            if any(overlaps(gs, ge, ps, pe) for (ps, pe) in pred_intervals_any):
                reach_para += 1
            # 技术匹配可达
            if any(overlaps(gs, ge, ps, pe) for (ps, pe) in pred_intervals_tech.get(gtech, [])):
                reach_tech += 1

    sr_para = reach_para / total_gold if total_gold else 0.0
    sr_tech = reach_tech / total_gold if total_gold else 0.0
    return {
        "total_gold": total_gold,
        "reach_para": reach_para,
        "reach_tech": reach_tech,
        "SR_para":    sr_para,
        "SR_tech":    sr_tech,
    }


print("✅ SR computation function ready")

## 步骤 4：执行 —— 9 个格子 + 汇总表

运行后：
- **SR_para** 列 → 直接填进论文 Table 5（技术无关，S2-C 的硬上界）
- 对照检查：每个格子的 `S2-C_recall ≤ SR_para` 应成立（S2-C recall 见 `实验结果汇总` 实验三）

In [ ]:
# 预加载金标准（每语言只读一次）
gold_cache = {}
for lang in LANGS:
    gb, n = load_spans_tsv(gold_path(lang))
    gold_cache[lang] = gb
    print(f"📂 gold[{lang}]: {n} 条金标准跨度，{len(gb)} 篇文章")

print("\n" + "=" * 78)
print(f"{'Lang':<5} {'Method':<10} {'total_gold':>10} {'reach_para':>11} {'SR_para':>9} {'SR_tech':>9}")
print("=" * 78)

rows = []
for lang in LANGS:
    for method in METHODS:
        pf = PARAGRAPH_FILES[(lang, method)]
        pred_by_article, _ = load_spans_tsv(pf)
        r = compute_span_recall(gold_cache[lang], pred_by_article)
        rows.append((lang, method, r))
        print(f"{lang:<5} {method:<10} {r['total_gold']:>10} {r['reach_para']:>11} "
              f"{r['SR_para']*100:>8.2f}% {r['SR_tech']*100:>8.2f}%")
    print("-" * 78)

In [ ]:
# ── 生成 Table 5 友好格式（SR_para 列）──
import pandas as pd

lang_disp = {"en": "EN", "po": "PL", "ru": "RU"}
tbl = {}
for lang, method, r in rows:
    tbl.setdefault(method, {})[f"{lang_disp[lang]} SR_para"] = round(r["SR_para"] * 100, 2)

df = pd.DataFrame(tbl).T
df = df[[f"{lang_disp[l]} SR_para" for l in LANGS]]  # 固定列顺序 EN/PL/RU
df.index.name = "Method"
print("📊 Table 5 用 SR_para（技术无关，% ，可直接填入论文）：\n")
print(df.to_string())

print("\n\n— 对照：当前论文 Table 5 的旧 SR（= Stage 1 × S2-A recall），用于核对 SR_para ≥ 旧值 —")
old_sr = {
    "Sup-FT":   {"EN": 22.65, "PL": 13.50, "RU": 22.33},
    "Prompt-A": {"EN": 8.05,  "PL": 8.07,  "RU": 12.99},
    "Iter-Ens": {"EN": 16.21, "PL": 18.12, "RU": 23.41},
}
print(pd.DataFrame(old_sr).T[["EN", "PL", "RU"]].to_string())

print("\n\n— 对照：实验三 S2-C recall（应满足 ≤ SR_para）—")
s2c_recall = {
    "Sup-FT":   {"EN": 24.88, "PL": 11.07, "RU": 18.94},
    "Prompt-A": {"EN": 19.32, "PL": 9.44,  "RU": 15.70},
    "Iter-Ens": {"EN": 22.27, "PL": 10.51, "RU": 18.94},
}
print(pd.DataFrame(s2c_recall).T[["EN", "PL", "RU"]].to_string())
print("\n✅ 若每格 SR_para ≥ S2-C recall，则 'hard upper bound' 成立，可放心写进论文。")

## 步骤 5：填表说明

1. 把上面 **SR_para 表**的 9 个数字填进论文 Table 5 的 `SR` 列（替换旧的 8.05% / 22.65% …）。
2. 检查最后一格的断言：**每个 SR_para 都 ≥ 对应的 S2-C recall**。
   - 若全部成立 → Sec 3.1 / 4.4 的 "hard upper bound" 表述可保留并改为技术无关定义。
   - 若某格 **SR_para < S2-C recall**（极小概率，可能因段落切分口径不同）→ 把我贴出来，
     说明 S2-C 处理的段落集合与 Stage 1 预测文件不完全一致，需要进一步对齐。
3. 顺手记下 Prompt-A 的 SR_para——它会明显高于旧的 8.05%，正好坐实
   "S2-C 因全技术预测能逼近段落上界"这个新卖点。